In [20]:
import pulp
from pulp import LpProblem, LpMinimize, LpVariable, lpSum
import numpy as np
import math
import time

In [21]:
start_time = time.time()

In [22]:
n = 200

In [23]:
scenarios = np.random.uniform(low=0.8, high=1.2, size=(n, 2)).tolist()

In [24]:
cumulative = np.random.uniform(low=0, high=1, size=n-1).tolist()
cumulative.extend([0, 1])
cumulative.sort()

In [25]:
probabilities = [cumulative[m + 1] - cumulative[m] for m in range(n)]
epsilon = 0.01

In [26]:
remaining_scenarios = scenarios
remaining_probabilities = probabilities
removed_scenarios = []
removed_probabilities = []

In [27]:
while True:
    remaining_n = range(len(remaining_scenarios))
    distances = [[math.inf if s1 == s2 else (remaining_scenarios[s1][0] - remaining_scenarios[s2][0])**2 + 
                  (remaining_scenarios[s1][1] - remaining_scenarios[s2][1])**2 for s2 in remaining_n] for s1 in remaining_n]

    min_distances = [min(distances[s]) for s in remaining_n]
    cumulative_removed_prob = sum(removed_probabilities)
    distance_probability = [(cumulative_removed_prob + remaining_probabilities[s]) * min_distances[s] for s in remaining_n]

    if min(distance_probability) > epsilon:
        break

    removing_index = distance_probability.index(min(distance_probability))
    removed_scenarios.append(remaining_scenarios.pop(removing_index))
    removed_probabilities.append(remaining_probabilities.pop(removing_index))

In [28]:
for removed_scenario, removed_prob in zip(removed_scenarios, removed_probabilities):
    distances_to_remaining = [(removed_scenario[0] - remaining_scenarios[remaining_s][0])**2 +
                              (removed_scenario[1] - remaining_scenarios[remaining_s][1])**2 for remaining_s in remaining_n]
    clustering_index = distances_to_remaining.index(min(distances_to_remaining))
    remaining_probabilities[clustering_index] += removed_prob

In [29]:
I = range(3)
J = range(4)
T = range(2)
S = range(len(remaining_scenarios))

In [30]:
PC = [150, 230, 260]  # Planting cost
BP = [238, 210]  # Buying price
SP = [170, 150, 36, 10]  # Selling price
MR = [200, 240]  # Minimum requirement
Default = [2.5, 3, 20]

In [31]:
farmer_problem = LpProblem("MultiStage_Farmer_Problem", LpMinimize)

In [32]:
X = LpVariable.matrix("X_it", (I, T), lowBound=0)
Y = LpVariable.matrix("Y_its", (I, T, S), lowBound=0)
W = LpVariable.matrix("W_jts", (J, T, S), lowBound=0)

In [33]:
farmer_problem += (lpSum(PC[i] * X[i][t] for i in I for t in T) +
                   lpSum(remaining_probabilities[s] * BP[i] * Y[i][t][s] for i in range(2) for t in T for s in S) -
                   lpSum(remaining_probabilities[s] * SP[j] * W[j][t][s] for j in J for t in T for s in S))

In [34]:
for t in T:
    farmer_problem += lpSum(X[i][t] for i in I) <= 500
    for s in S:
        for i in range(2):
            farmer_problem += (remaining_scenarios[s][t] * Default[i]) * X[i][t] + Y[i][t][s] - W[i][t][s] >= MR[i]
        farmer_problem += (remaining_scenarios[s][t] * Default[2]) * X[2][t] - W[2][t][s] - W[3][t][s] >= 0
        farmer_problem += W[2][t][s] <= 6000

farmer_problem += X[2][0] + X[2][1] <= 500

In [35]:
farmer_problem.solve()

1

In [36]:
objective_value = round(farmer_problem.objective.value(), 2)
print(f'Objective Value is {objective_value}')

Objective Value is -213865.85


In [37]:
for v in farmer_problem.variables():
    if v.name.startswith("X_it"):
        print(f'{v.name} = {round(v.varValue, 2)}')

X_it_0_0 = 157.99
X_it_0_1 = 164.18
X_it_1_0 = 93.44
X_it_1_1 = 84.4
X_it_2_0 = 248.57
X_it_2_1 = 251.43


In [38]:
print(f"Run time is: {round(time.time() - start_time, 2)} seconds")

Run time is: 1.43 seconds
